In [10]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


def prepare_data(df, target_col='class'):

    print("\n===== DATA PREPARATION STARTED =====")

    # Verify required columns
    required_cols = [
        target_col,
        'user_id',
        'device_id',
        'ip_address',
        'signup_time',
        'purchase_time'
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    # Separate features and target
    X = df.drop(
        [
            target_col,
            'user_id',
            'device_id',
            'ip_address',
            'signup_time',
            'purchase_time'
        ],
        axis=1
    )

    y = df[target_col]

    print(f"\nFeature shape: {X.shape}")
    print(f"Target shape: {y.shape}")

    print("\nClass distribution before split:")
    print(y.value_counts())

    # Identify column types
    numeric = X.select_dtypes(include=np.number).columns.tolist()
    categorical = X.select_dtypes(include=['object', 'category']).columns.tolist()

    print("\nNumeric columns:")
    print(numeric)

    print("\nCategorical columns:")
    print(categorical)

    # Preprocessing pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
        ]
    )

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    print("\nTrain shape:", X_train.shape)
    print("Test shape:", X_test.shape)

    # Transform training data
    X_train_transformed = preprocessor.fit_transform(X_train)

    # Apply SMOTE only on training set
    smote = SMOTE(random_state=42)

    X_train_resampled, y_train_resampled = smote.fit_resample(
        X_train_transformed,
        y_train
    )

    print("\nClass distribution BEFORE SMOTE:")
    print(y_train.value_counts())

    print("\nClass distribution AFTER SMOTE:")
    print(pd.Series(y_train_resampled).value_counts())

    print("\nResampled training shape:", X_train_resampled.shape)

    return (
        X_train_resampled,
        X_test,
        y_train_resampled,
        y_test,
        preprocessor
    )


if __name__ == "__main__":

    print("Current Working Directory:")
    print(os.getcwd())

    file_path = "../data/processed/cleaned_fraud_data.csv"

    print("\nChecking file existence:")
    print(os.path.exists(file_path))

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path)

    print("\nDataset Loaded Successfully")

    print("Dataset Shape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nTarget Distribution:")
    print(df["class"].value_counts())

    X_train_res, X_test, y_train_res, y_test, preprocessor = prepare_data(df)

    print("\n===== PREPROCESSING COMPLETED SUCCESSFULLY =====")

Current Working Directory:
c:\Users\PC\fraud-detection\notebooks

Checking file existence:
True

Dataset Loaded Successfully
Dataset Shape:
(151112, 11)

Columns:
['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'source', 'browser', 'sex', 'age', 'ip_address', 'class']

Target Distribution:
class
0    136961
1     14151
Name: count, dtype: int64

===== DATA PREPARATION STARTED =====

Feature shape: (151112, 5)
Target shape: (151112,)

Class distribution before split:
class
0    136961
1     14151
Name: count, dtype: int64

Numeric columns:
['purchase_value', 'age']

Categorical columns:
['source', 'browser', 'sex']

Train shape: (120889, 5)
Test shape: (30223, 5)

Class distribution BEFORE SMOTE:
class
0    109568
1     11321
Name: count, dtype: int64

Class distribution AFTER SMOTE:
class
0    109568
1    109568
Name: count, dtype: int64

Resampled training shape: (219136, 12)

===== PREPROCESSING COMPLETED SUCCESSFULLY =====
